<a href="https://colab.research.google.com/github/siyaagarwal2005/sepsis-prediction-using-LSTM/blob/main/2_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook II — Preprocessing

---

**Project:** AI-Based Sepsis Early Warning System  
**Dataset:** PhysioNet Challenge 2019 (pre-merged CSV)  
**Author:** *Siya Agarwal*  
**Date:** *05 March, 2026*

---

### Purpose
Transform the raw dataset into a clean, model-ready form. This notebook executes the full preprocessing pipeline in five stages:
1. Load & subset to 17 selected features
2. Sort data by patient and time
3. Patient-wise train/test split
4. Missing value handling — forward-fill (LOCF) then median imputation
5. Feature scaling — StandardScaler fit on train set only

### Inputs
- `/content/drive/MyDrive/sepsis_project/data/sepsis_dataset.csv` — raw merged dataset

### Outputs
- `/content/drive/MyDrive/sepsis_project/processed_data/train_processed.csv`
- `/content/drive/MyDrive/sepsis_project/processed_data/test_processed.csv`

### Key Design Decisions
| Decision | Choice | Reason |
|----------|--------|--------|
| Train/test split | Patient-wise | Prevents patient data leaking across sets |
| Missing value strategy | LOCF → median | Preserves temporal continuity; median is robust to outliers |
| Scaler fit | Train set only | Prevents data leakage from test set into scaling parameters |
| Scaler type | StandardScaler | Zero mean, unit variance — standard for neural networks |

### Next Notebook
- `III_sequence_engineering.ipynb` — sliding window sequences for LSTM input

---
## Section 0 — Setup

Mount Google Drive and import all required libraries.  
All project paths are anchored to `BASE_PATH` — change one line, nothing else breaks.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.preprocessing import StandardScaler
from IPython.display import display

# ── Display settings ───────────────────────────────────────────────────────────
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')

# ── Project paths ──────────────────────────────────────────────────────────────
BASE_PATH       = '/content/drive/MyDrive/sepsis_project/'
DATA_PATH       = BASE_PATH + 'data/sepsis_dataset.csv'
PROCESSED_PATH  = BASE_PATH + 'processed_data/'

os.makedirs(PROCESSED_PATH, exist_ok=True)  # create folder if it doesn't exist

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_SEED = 42

print('Setup complete.')
print(f'Data path      : {DATA_PATH}')
print(f'Output path    : {PROCESSED_PATH}')

Setup complete.
Data path      : /content/drive/MyDrive/sepsis_project/data/sepsis_dataset.csv
Output path    : /content/drive/MyDrive/sepsis_project/processed_data/


---
## Section 1 — Load & Subset to Selected Features

We load the full raw dataset and immediately reduce it to our 17 selected features.  
This reduces memory from ~546 MB to ~120 MB and removes all noise columns.

We also drop the `Unnamed: 0` and `Hour` artefact columns if present.

In [ ]:
df = pd.read_csv(DATA_PATH)

# ── Drop merge artefacts ───────────────────────────────────────────────────────
cols_to_drop = [c for c in ['Unnamed: 0', 'Hour'] if c in df.columns]
if cols_to_drop:
    df.drop(columns=cols_to_drop, inplace=True)
    print(f'Dropped artefact columns: {cols_to_drop}')

# ── Select 17 features ────────────────────────────────────────────────────────
# 12 clinical features (7 vitals + 4 labs + 1 composite vital)
# 2 demographic features
# 1 time axis (ICULOS)
# 1 patient identifier (Patient_ID — used for grouping, not as a model feature)
# 1 target label (SepsisLabel)
SELECTED_FEATURES = [
    'HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',  # vitals
    'Lactate', 'WBC', 'Creatinine', 'BUN', 'pH',          # labs
    'Age', 'Gender',                                        # demographics
    'ICULOS',                                               # time axis
    'Patient_ID', 'SepsisLabel'                             # meta
]

df_sel = df[SELECTED_FEATURES].copy()

print(f'Original shape   : {df.shape}')
print(f'Subset shape     : {df_sel.shape}')   # expect (1552210, 17)
print(f'\nMissing values before imputation:')
display(df_sel.isnull().mean().mul(100).round(2).rename('missing_pct').sort_values(ascending=False))

Dropped artefact columns: ['Unnamed: 0', 'Hour']
Original shape   : (1552210, 42)
Subset shape     : (1552210, 17)

Missing values before imputation:


,missing_pct
Lactate,97.3300
Creatinine,93.9000
WBC,93.5900
BUN,93.1300
pH,93.0700
Temp,66.1600
DBP,31.3500
Resp,15.3500
SBP,14.5800
O2Sat,13.0600


---
## Section 2 — Sort by Patient and Time

Before any imputation or splitting, we sort the dataframe by `Patient_ID` then `ICULOS`.

This is **mandatory** before forward-fill. If rows are out of temporal order, forward-fill
will propagate values in the wrong direction — corrupting every patient's time series.

Think of it like reading a patient's medical chart: the pages must be in order before
you can carry the last observation forward.

In [ ]:
df_sel = df_sel.sort_values(['Patient_ID', 'ICULOS']).reset_index(drop=True)

# Verify sort: first patient's rows should have ICULOS = 1, 2, 3...
first_patient = df_sel['Patient_ID'].iloc[0]
first_patient_rows = df_sel[df_sel['Patient_ID'] == first_patient]

print(f'First patient in dataset: {first_patient}')
print(f'Their ICULOS values: {first_patient_rows["ICULOS"].values}')
print(f'\nDataframe head after sort:')
display(df_sel.head(10))

First patient in dataset: 1
Their ICULOS values: [ 1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18 19 20 21 22 23 24
 25 26 27 28 29 30 31 32 33 34 35 36 37 38 39 40 41 42 43 44 45 46 47 48
 49 50 51 52 53 54]

Dataframe head after sort:


,HR,O2Sat,Temp,SBP,MAP,DBP,Resp,Lactate,WBC,Creatinine,BUN,pH,Age,Gender,ICULOS,Patient_ID,SepsisLabel
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,83.1400,0,1,1,0
1,97.0000,95.0000,NaN,98.0000,75.3300,NaN,19.0000,NaN,NaN,NaN,NaN,NaN,83.1400,0,2,1,0
2,89.0000,99.0000,NaN,122.0000,86.0000,NaN,22.0000,NaN,NaN,NaN,NaN,NaN,83.1400,0,3,1,0
3,90.0000,95.0000,NaN,NaN,NaN,NaN,30.0000,NaN,NaN,NaN,NaN,7.3600,83.1400,0,4,1,0
4,103.0000,88.5000,NaN,122.0000,91.3300,NaN,24.5000,NaN,NaN,NaN,NaN,NaN,83.1400,0,5,1,0
5,110.0000,91.0000,NaN,NaN,NaN,NaN,22.0000,NaN,NaN,NaN,NaN,NaN,83.1400,0,6,1,0
6,108.0000,92.0000,36.1100,123.0000,77.0000,NaN,29.0000,NaN,NaN,NaN,NaN,NaN,83.1400,0,7,1,0
7,106.0000,90.5000,NaN,93.0000,76.3300,NaN,29.0000,NaN,NaN,NaN,NaN,NaN,83.1400,0,8,1,0
8,104.0000,95.0000,NaN,133.0000,88.3300,NaN,26.0000,NaN,NaN,NaN,NaN,NaN,83.1400,0,9,1,0
9,102.0000,91.0000,NaN,134.0000,87.3300,NaN,30.0000,NaN,NaN,NaN,NaN,NaN,83.1400,0,10,1,0


---
## Section 3 — Patient-Wise Train/Test Split

We split patients into train (80%) and test (20%) **before** any imputation or scaling.

### Why patient-wise, not row-wise?
If we split rows randomly, the same patient can appear in both train and test sets.
The model would then see future hours of a patient during training and earlier hours
of the same patient during testing — effectively memorising patients rather than
learning generalisable patterns.

Patient-wise split guarantees: **a patient the model has never seen during training
is what gets evaluated at test time.** This is the only clinically valid evaluation setup.

### Why split before imputation?
If we impute first (using global medians computed on all data), the test set
influences the imputation values — this is **data leakage**. Medians must be
computed on the train set only, then applied to both.

In [ ]:
import numpy as np

# ── Get all unique patients and their sepsis status ────────────────────────────
patient_labels = df_sel.groupby('Patient_ID')['SepsisLabel'].max().reset_index()
patient_labels.columns = ['Patient_ID', 'is_septic']

# ── Stratified patient-wise split ─────────────────────────────────────────────
# Stratify by sepsis status to preserve ~7.27% sepsis rate in both sets.
# Without stratification, random chance could put most septic patients in one split.
from sklearn.model_selection import train_test_split

train_patients, test_patients = train_test_split(
    patient_labels['Patient_ID'],
    test_size=0.20,
    random_state=RANDOM_SEED,
    stratify=patient_labels['is_septic']  # preserve sepsis rate in both sets
)

train_set = patient_labels[patient_labels['Patient_ID'].isin(train_patients)]
test_set  = patient_labels[patient_labels['Patient_ID'].isin(test_patients)]

df_train = df_sel[df_sel['Patient_ID'].isin(train_patients)].copy()
df_test  = df_sel[df_sel['Patient_ID'].isin(test_patients)].copy()

# ── Verify split quality ───────────────────────────────────────────────────────
print('=== Train/Test Split Summary ===')
print(f'Train patients   : {len(train_patients):,}  ({len(train_patients)/len(patient_labels)*100:.1f}%)')
print(f'Test  patients   : {len(test_patients):,}   ({len(test_patients)/len(patient_labels)*100:.1f}%)')
print(f'Train rows       : {len(df_train):,}')
print(f'Test  rows       : {len(df_test):,}')

train_sepsis_rate = train_set['is_septic'].mean() * 100
test_sepsis_rate  = test_set['is_septic'].mean() * 100
print(f'\nTrain sepsis rate: {train_sepsis_rate:.2f}%')   # should be ~7.27%
print(f'Test  sepsis rate: {test_sepsis_rate:.2f}%')    # should be ~7.27%
print('\nSepsis rates are consistent — stratification successful.' if abs(train_sepsis_rate - test_sepsis_rate) < 0.5
      else 'WARNING: Sepsis rates differ significantly between splits.')

=== Train/Test Split Summary ===
Train patients   : 32,268  (80.0%)
Test  patients   : 8,068   (20.0%)
Train rows       : 1,240,596
Test  rows       : 311,614

Train sepsis rate: 7.27%
Test  sepsis rate: 7.26%

Sepsis rates are consistent — stratification successful.


---
## Section 4 — Missing Value Handling

We use a two-step imputation strategy:

### Step 4a — Forward Fill Per Patient (LOCF)
LOCF (Last Observation Carried Forward) mimics real clinical practice: if no new
measurement has been taken, the most recent known value is assumed to still hold.

This is applied **per patient** (not globally) to prevent one patient's values
from filling another patient's gaps.

**Why values remain after forward-fill:**  
Forward-fill cannot look backwards. If a lab was never measured in a patient's
first few ICU hours, those leading NaNs persist — this is clinically correct behaviour,
not a bug.

### Step 4b — Median Imputation for Remaining NaNs
For NaNs that survive forward-fill (typically at the start of a patient's record),
we fill with the **train-set median** of each column.

Why median over mean? Median is robust to outliers, which are common in ICU lab values.
Why train-set median? Computing medians on all data would let test data influence
imputation — a form of data leakage.

In [ ]:
# ── Step 4a: Forward fill per patient ─────────────────────────────────────────
# groupby Patient_ID, apply ffill within each group, then reset the index.
df_train = df_train.groupby('Patient_ID', group_keys=False).apply(lambda x: x.ffill())
df_test  = df_test.groupby('Patient_ID', group_keys=False).apply(lambda x: x.ffill())

df_train.reset_index(drop=True, inplace=True)
df_test.reset_index(drop=True, inplace=True)

remaining_train = df_train.isnull().sum().sum()
remaining_test  = df_test.isnull().sum().sum()

print('=== After Forward-Fill ===')
print(f'Remaining NaNs in train : {remaining_train:,}')
print(f'Remaining NaNs in test  : {remaining_test:,}')
print('(Some NaNs expected — these are leading gaps at the start of patient records)')

/tmp/ipykernel_232/1285038968.py:3: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_train = df_train.groupby('Patient_ID', group_keys=False).apply(lambda x: x.ffill())


=== After Forward-Fill ===
Remaining NaNs in train : 2,877,671
Remaining NaNs in test  : 719,113
(Some NaNs expected — these are leading gaps at the start of patient records)


/tmp/ipykernel_232/1285038968.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_test  = df_test.groupby('Patient_ID', group_keys=False).apply(lambda x: x.ffill())


In [ ]:
# ── Step 4b: Median imputation for remaining NaNs ─────────────────────────────
# Compute medians from train set ONLY.
# Apply to both train and test — test never influences the imputation values.

FEATURE_COLS = ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp',
                'Lactate', 'WBC', 'Creatinine', 'BUN', 'pH',
                'Age', 'Gender', 'ICULOS']

train_medians = df_train[FEATURE_COLS].median()

df_train[FEATURE_COLS] = df_train[FEATURE_COLS].fillna(train_medians)
df_test[FEATURE_COLS]  = df_test[FEATURE_COLS].fillna(train_medians)  # use train medians

# ── Final NaN check ────────────────────────────────────────────────────────────
final_train_nans = df_train.isnull().sum().sum()
final_test_nans  = df_test.isnull().sum().sum()

print('=== After Median Imputation ===')
print(f'Remaining NaNs in train : {final_train_nans}')   # must be 0
print(f'Remaining NaNs in test  : {final_test_nans}')    # must be 0

if final_train_nans == 0 and final_test_nans == 0:
    print('\nAll missing values resolved. Dataset is complete.')
else:
    print('\nWARNING: NaNs remain. Investigate before proceeding.')

print(f'\nTrain medians used for imputation:')
display(train_medians.round(4))

=== After Median Imputation ===
Remaining NaNs in train : 0
Remaining NaNs in test  : 0

All missing values resolved. Dataset is complete.

Train medians used for imputation:


,0
HR,83.0000
O2Sat,98.0000
Temp,36.8300
SBP,121.0000
MAP,81.0000
DBP,62.0000
Resp,18.0000
Lactate,1.5800
WBC,10.3000
Creatinine,0.9000


---
## Section 5 — Feature Scaling

Neural networks are sensitive to feature scale. Our features currently span very different ranges:
- `Age` ~ 20–100 (years)
- `HR` ~ 40–200 (beats per minute)
- `Creatinine` ~ 0.3–10 (mg/dL)
- `Lactate` ~ 0.5–20 (mmol/L)

Without scaling, features with large values dominate gradient updates during training,
causing the LSTM to effectively ignore features with smaller ranges.

### StandardScaler
Transforms each feature to **zero mean and unit variance**:  
`z = (x - mean) / std`

**Critical rule:** Fit the scaler on the **train set only**.  
Then apply (transform) to both train and test.  
Fitting on all data would leak test set statistics into training — the model would
indirectly know the scale of data it has never seen.

We scale only the 15 feature columns — **not** `Patient_ID` or `SepsisLabel`.

In [ ]:
scaler = StandardScaler()

# ── Fit on train features only ─────────────────────────────────────────────────
scaler.fit(df_train[FEATURE_COLS])

# ── Transform both sets ────────────────────────────────────────────────────────
df_train[FEATURE_COLS] = scaler.transform(df_train[FEATURE_COLS])
df_test[FEATURE_COLS]  = scaler.transform(df_test[FEATURE_COLS])

# ── Verify scaling ─────────────────────────────────────────────────────────────
# After scaling, train feature means should be ~0 and stds should be ~1
print('=== Scaling Verification (Train Set) ===')
scale_check = pd.DataFrame({
    'mean_after_scale' : df_train[FEATURE_COLS].mean().round(4),
    'std_after_scale'  : df_train[FEATURE_COLS].std().round(4)
})
display(scale_check)
print('\nMeans should be ~0.0 and stds should be ~1.0')

=== Scaling Verification (Train Set) ===


,mean_after_scale,std_after_scale
HR,-0.0000,1.0000
O2Sat,0.0000,1.0000
Temp,0.0000,1.0000
SBP,0.0000,1.0000
MAP,-0.0000,1.0000
DBP,-0.0000,1.0000
Resp,0.0000,1.0000
Lactate,0.0000,1.0000
WBC,0.0000,1.0000
Creatinine,-0.0000,1.0000



Means should be ~0.0 and stds should be ~1.0


---
## Section 6 — Save Processed Data

We save the processed train and test sets to the `processed_data/` folder on Google Drive.
The next notebook (`III_sequence_engineering.ipynb`) loads directly from these files —
it does not re-run any preprocessing steps.

**What each saved file contains:**
- 15 scaled feature columns
- `Patient_ID` (for grouping into sequences)
- `SepsisLabel` (target variable)
- Rows sorted by Patient_ID then ICULOS — temporal order preserved

In [ ]:
TRAIN_OUT = PROCESSED_PATH + 'train_processed.csv'
TEST_OUT  = PROCESSED_PATH + 'test_processed.csv'

df_train.to_csv(TRAIN_OUT, index=False)
df_test.to_csv(TEST_OUT,  index=False)

print('=== Files Saved ===')
print(f'Train : {TRAIN_OUT}')
print(f'        Shape: {df_train.shape}')
print(f'Test  : {TEST_OUT}')
print(f'        Shape: {df_test.shape}')
print(f'\nColumns saved: {list(df_train.columns)}')

=== Files Saved ===
Train : /content/drive/MyDrive/sepsis_project/processed_data/train_processed.csv
        Shape: (1240596, 17)
Test  : /content/drive/MyDrive/sepsis_project/processed_data/test_processed.csv
        Shape: (311614, 17)

Columns saved: ['HR', 'O2Sat', 'Temp', 'SBP', 'MAP', 'DBP', 'Resp', 'Lactate', 'WBC', 'Creatinine', 'BUN', 'pH', 'Age', 'Gender', 'ICULOS', 'Patient_ID', 'SepsisLabel']


---
## Section 7 — Preprocessing Verification

A final end-to-end sanity check before handing data to the next notebook.

We verify:
- No NaNs remain in either set
- Sepsis rates are preserved in both sets
- Patient overlap between train and test is zero
- Row counts are consistent

In [ ]:
print('=== Final Preprocessing Verification ===')

# 1. NaN check
print(f'NaNs in train            : {df_train.isnull().sum().sum()}')   # must be 0
print(f'NaNs in test             : {df_test.isnull().sum().sum()}')    # must be 0

# 2. Sepsis rate preservation
train_sr = df_train.groupby('Patient_ID')['SepsisLabel'].max().mean() * 100
test_sr  = df_test.groupby('Patient_ID')['SepsisLabel'].max().mean() * 100
print(f'Train patient sepsis rate: {train_sr:.2f}%')   # expect ~7.27%
print(f'Test  patient sepsis rate: {test_sr:.2f}%')    # expect ~7.27%

# 3. Patient overlap — must be zero
overlap = set(df_train['Patient_ID']).intersection(set(df_test['Patient_ID']))
print(f'Patient overlap          : {len(overlap)}')    # must be 0

# 4. Row count check
print(f'Train rows               : {len(df_train):,}')
print(f'Test  rows               : {len(df_test):,}')
print(f'Total rows               : {len(df_train) + len(df_test):,}')  # should match original

print('\nPreprocessing complete. Ready for III_sequence_engineering.ipynb')

=== Final Preprocessing Verification ===
NaNs in train            : 0
NaNs in test             : 0
Train patient sepsis rate: 7.27%
Test  patient sepsis rate: 7.26%
Patient overlap          : 0
Train rows               : 1,240,596
Test  rows               : 311,614
Total rows               : 1,552,210

Preprocessing complete. Ready for III_sequence_engineering.ipynb


---
## Summary

### What This Notebook Did

**Stage 1 — Feature Subset**  
Reduced 44 raw columns to 17 clinically selected features. Dropped `Unnamed: 0` and `Hour` artefacts.

**Stage 2 — Temporal Sort**  
Sorted all rows by `Patient_ID` then `ICULOS` to ensure correct chronological order per patient.

**Stage 3 — Patient-Wise Split**  
Split 40,336 patients into 80% train / 20% test using stratified sampling to preserve the ~7.27% sepsis rate in both sets. No patient appears in both sets.

**Stage 4 — Missing Value Handling**  
Applied forward-fill (LOCF) per patient to propagate last known measurements forward. Filled remaining leading NaNs with train-set medians. Test set used train medians — no leakage.

**Stage 5 — Feature Scaling**  
Fitted StandardScaler on train features only. Transformed both train and test. Features now have zero mean and unit variance.

---
## Next Steps — `III_sequence_engineering.ipynb`

1. Load `train_processed.csv` and `test_processed.csv`
2. Define sliding window of 6 hours (`WINDOW_SIZE = 6`)
3. Generate sequences per patient: each sample = 6 hours of features → 1 label
4. Save `X_train`, `y_train`, `X_test`, `y_test` as `.npy` arrays
5. Verify output shape: `X.shape = (n_sequences, 6, 15)`